In [5]:
!pip install mediapipe opencv-python scikit-learn numpy matplotlib tqdm fastapi uvicorn pydantic
print('\n✅ All packages installed!')


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

✅ All packages installed!


In [7]:
import os
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm import tqdm
import urllib.request

# ✏️ Your dataset path (folders are lowercase a, b, c... inside)
DATA_DIR      = '/Users/bhavitejareddy/Downloads/dataset - Gesture Speech'
MAX_PER_CLASS = 350
OUTPUT_PATH   = 'data/keypoints.npz'

os.makedirs('data', exist_ok=True)

# ── Download the hand landmark model file if not already present ──────────
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Downloading hand landmarker model (~9MB)...')
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Downloaded:', MODEL_PATH)
else:
    print('Model file already exists:', MODEL_PATH)

LETTERS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')

def find_folder(data_dir, letter):
    for name in [letter.upper(), letter.lower()]:
        path = os.path.join(data_dir, name)
        if os.path.isdir(path):
            return path
    return None

def normalize(kps):
    pts = kps.reshape(21, 3)
    pts = pts - pts[0]                   # wrist to origin
    scale = np.linalg.norm(pts[9])
    if scale > 0:
        pts /= scale
    return pts.flatten()

# ── Build detector using new Tasks API ────────────────────────────────────
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    running_mode=mp_vision.RunningMode.IMAGE,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

X, y = [], []
failed = 0

for letter in LETTERS:
    folder = find_folder(DATA_DIR, letter)
    if folder is None:
        print(f'  ⚠  No folder for {letter}, skipping')
        continue

    imgs = [f for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    imgs = imgs[:MAX_PER_CLASS]
    extracted = 0

    for fname in tqdm(imgs, desc=f'{letter} ({len(imgs)} imgs)', leave=False):
        img_path = os.path.join(folder, fname)
        img_bgr  = cv2.imread(img_path)
        if img_bgr is None:
            failed += 1
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        result   = detector.detect(mp_image)
        if not result.hand_landmarks:
            failed += 1
            continue
        lms = result.hand_landmarks[0]
        kps = np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()
        kps = normalize(kps)
        X.append(kps)
        y.append(letter)
        extracted += 1

    print(f'  {letter}: {extracted}/{len(imgs)} extracted')

detector.close()

X = np.array(X, dtype=np.float32)
y = np.array(y)
np.savez_compressed(OUTPUT_PATH, X=X, y=y)

print(f'\n✅ Saved {len(X)} samples → {OUTPUT_PATH}')
print(f'   Failed (no hand detected) : {failed}')
print(f'   Classes found             : {sorted(set(y))}')
print(f'   Shape                     : X={X.shape}  y={y.shape}')

I0000 00:00:1777548041.601927 10956733 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1
W0000 00:00:1777548041.617927 10956740 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777548041.622998 10956740 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Downloaded: hand_landmarker.task


  A: 312/350 extracted


  B: 267/350 extracted


  C: 308/350 extracted


...

  W: 315/350 extracted


  X: 299/350 extracted


  Y: 325/350 extracted


  Z: 77/350 extracted

✅ Saved 7344 samples → data/keypoints.npz
   Failed (no hand detected) : 1756
   Classes found             : [np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('J'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('O'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('X'), np.str_('Y'), np.str_('Z')]
   Shape                     : X=(7344, 63)  y=(7344,)


E0000 00:00:1777548268.451559 10956734 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-04-30T17:06:41.857349+05:30
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


In [8]:
import os, pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

os.makedirs("models", exist_ok=True)

# ── Load keypoints saved in Step 1 ────────────────────────────────────────
print("Loading keypoints...")
data = np.load("data/keypoints.npz")
X, y = data["X"], data["y"]
print(f"  Samples : {len(X)}")
print(f"  Features: {X.shape[1]}  (21 landmarks × xyz)")
print(f"  Classes : {sorted(set(y))}")

Loading keypoints...
  Samples : 7344
  Features: 63  (21 landmarks × xyz)
  Classes : [np.str_('A'), np.str_('B'), np.str_('C'), np.str_('D'), np.str_('E'), np.str_('F'), np.str_('G'), np.str_('H'), np.str_('I'), np.str_('J'), np.str_('K'), np.str_('L'), np.str_('M'), np.str_('N'), np.str_('O'), np.str_('P'), np.str_('Q'), np.str_('R'), np.str_('S'), np.str_('T'), np.str_('U'), np.str_('V'), np.str_('W'), np.str_('X'), np.str_('Y'), np.str_('Z')]


In [9]:
# ── Encode labels & split ──────────────────────────────────────────────────
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"Label map: {dict(zip(le.classes_, le.transform(le.classes_)))}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.15, stratify=y_enc, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

Label map: {np.str_('A'): np.int64(0), np.str_('B'): np.int64(1), np.str_('C'): np.int64(2), np.str_('D'): np.int64(3), np.str_('E'): np.int64(4), np.str_('F'): np.int64(5), np.str_('G'): np.int64(6), np.str_('H'): np.int64(7), np.str_('I'): np.int64(8), np.str_('J'): np.int64(9), np.str_('K'): np.int64(10), np.str_('L'): np.int64(11), np.str_('M'): np.int64(12), np.str_('N'): np.int64(13), np.str_('O'): np.int64(14), np.str_('P'): np.int64(15), np.str_('Q'): np.int64(16), np.str_('R'): np.int64(17), np.str_('S'): np.int64(18), np.str_('T'): np.int64(19), np.str_('U'): np.int64(20), np.str_('V'): np.int64(21), np.str_('W'): np.int64(22), np.str_('X'): np.int64(23), np.str_('Y'): np.int64(24), np.str_('Z'): np.int64(25)}

Train: 6242 | Test: 1102


In [11]:
# ── Train MLP ─────────────────────────────────────────────────────────────
print("Training MLP...")
mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(512, 256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=64,
        learning_rate="adaptive",
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=True,
    ))
])
mlp_pipe.fit(X_train, y_train)
mlp_preds = mlp_pipe.predict(X_test)
mlp_acc   = accuracy_score(y_test, mlp_preds)
print(f"\n✅ MLP Test Accuracy: {mlp_acc*100:.1f}%")

Training MLP...
Iteration 1, loss = 1.50785501
Validation score: 0.731200
Iteration 2, loss = 0.64695546
Validation score: 0.817600
Iteration 3, loss = 0.46499634
Validation score: 0.840000
Iteration 4, loss = 0.37397362
Validation score: 0.878400
Iteration 5, loss = 0.32722545
Validation score: 0.888000
Iteration 6, loss = 0.26510665
Validation score: 0.929600
Iteration 7, loss = 0.21478126
Validation score: 0.913600
Iteration 8, loss = 0.19023235
Validation score: 0.936000
Iteration 9, loss = 0.16918776
Validation score: 0.928000
Iteration 10, loss = 0.14521873
Validation score: 0.952000
Iteration 11, loss = 0.12910584
Validation score: 0.942400
Iteration 12, loss = 0.14026392
Validation score: 0.916800
Iteration 13, loss = 0.12186646
Validation score: 0.945600
Iteration 14, loss = 0.12197506
Validation score: 0.947200
Iteration 15, loss = 0.11477013
Validation score: 0.948800
Iteration 16, loss = 0.10450986
Validation score: 0.945600
Iteration 17, loss = 0.09643380
Validation score:

In [12]:
# ── Train SVM ─────────────────────────────────────────────────────────────
print("Training SVM (RBF)...")
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42))
])
svm_pipe.fit(X_train, y_train)
svm_preds = svm_pipe.predict(X_test)
svm_acc   = accuracy_score(y_test, svm_preds)
print(f"✅ SVM Test Accuracy: {svm_acc*100:.1f}%")

# ── Pick winner ───────────────────────────────────────────────────────────
best_model = mlp_pipe if mlp_acc >= svm_acc else svm_pipe
best_name  = "MLP"    if mlp_acc >= svm_acc else "SVM"
best_preds = mlp_preds if mlp_acc >= svm_acc else svm_preds
best_acc   = max(mlp_acc, svm_acc)
print(f"\n🏆 Best model: {best_name} ({best_acc*100:.1f}%)")

Training SVM (RBF)...
✅ SVM Test Accuracy: 97.6%

🏆 Best model: MLP (98.7%)


In [13]:
# ── Classification report ─────────────────────────────────────────────────
report = classification_report(y_test, best_preds, target_names=le.classes_)
print(report)

with open("models/alphabet_report.txt", "w") as f:
    f.write(f"Best model  : {best_name}\n")
    f.write(f"MLP accuracy: {mlp_acc*100:.2f}%\n")
    f.write(f"SVM accuracy: {svm_acc*100:.2f}%\n\n")
    f.write(report)

              precision    recall  f1-score   support

           A       1.00      1.00      1.00        47
           B       0.98      1.00      0.99        40
           C       1.00      1.00      1.00        46
           D       0.95      0.97      0.96        40
           E       0.98      0.94      0.96        51
           F       1.00      1.00      1.00        47
           G       1.00      1.00      1.00        36
           H       1.00      0.97      0.99        38
           I       1.00      1.00      1.00        48
           J       0.98      1.00      0.99        45
           K       0.98      0.98      0.98        48
           L       1.00      0.98      0.99        45
           M       1.00      1.00      1.00        41
           N       1.00      1.00      1.00        47
           O       0.98      0.98      0.98        48
           P       0.90      0.95      0.93        40
           Q       1.00      0.98      0.99        41
           R       1.00    

In [14]:
# ── Confusion matrix ──────────────────────────────────────────────────────
cm   = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=45)
ax.set_title(f"ISL Alphabet — {best_name} Confusion Matrix\nTest Accuracy: {best_acc*100:.1f}%", fontsize=14)
plt.tight_layout()
plt.savefig("models/confusion_matrix.png", dpi=150)
plt.show()
print("Saved: models/confusion_matrix.png")

# ── Per-class accuracy bar chart ──────────────────────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)
fig2, ax2 = plt.subplots(figsize=(14, 5))
colors = ["#148A50" if a >= 0.9 else "#F06A2A" if a >= 0.7 else "#E74C3C" for a in per_class_acc]
bars   = ax2.bar(le.classes_, per_class_acc * 100, color=colors, edgecolor="white", linewidth=0.5)
ax2.axhline(90, color="green",  linestyle="--", linewidth=1, alpha=0.6, label="90% threshold")
ax2.axhline(70, color="orange", linestyle="--", linewidth=1, alpha=0.6, label="70% threshold")
ax2.set_ylim(0, 105)
ax2.set_xlabel("ISL Letter", fontsize=12)
ax2.set_ylabel("Accuracy (%)", fontsize=12)
ax2.set_title(f"Per-Class Accuracy — ISL Alphabet ({best_name})", fontsize=13)
ax2.legend()
for bar, acc in zip(bars, per_class_acc):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"{acc*100:.0f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig("models/per_class_accuracy.png", dpi=150)
plt.show()
print("Saved: models/per_class_accuracy.png")

Saved: models/confusion_matrix.png
Saved: models/per_class_accuracy.png


/var/folders/47/8ksx2fqj2cs06pgbq_30n2900000gn/T/ipykernel_3721/2915746844.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/47/8ksx2fqj2cs06pgbq_30n2900000gn/T/ipykernel_3721/2915746844.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# ── Save BOTH models separately ───────────────────────────────────────────

# Save SVM (forced — we want to test SVM specifically)
svm_data = {
    "model"         : svm_pipe,
    "label_encoder" : le,
    "model_type"    : 'SVM',
    "accuracy"      : svm_acc,
    "num_classes"   : len(le.classes_),
    "feature_dim"   : X.shape[1],
    "classes"       : list(le.classes_),
}
with open('models/alphabet_model_svm.pkl', 'wb') as f:
    pickle.dump(svm_data, f)
print(f'✅ Saved SVM  → models/alphabet_model_svm.pkl  ({svm_acc*100:.1f}%)')

# Save MLP
mlp_data = {
    "model"         : mlp_pipe,
    "label_encoder" : le,
    "model_type"    : 'MLP',
    "accuracy"      : mlp_acc,
    "num_classes"   : len(le.classes_),
    "feature_dim"   : X.shape[1],
    "classes"       : list(le.classes_),
}
with open('models/alphabet_model_mlp.pkl', 'wb') as f:
    pickle.dump(mlp_data, f)
print(f'✅ Saved MLP  → models/alphabet_model_mlp.pkl  ({mlp_acc*100:.1f}%)')

# Also save SVM as the default alphabet_model.pkl used by the server
import shutil
shutil.copy('models/alphabet_model_svm.pkl', 'models/alphabet_model.pkl')
print(f'\n🎯 Default model set to SVM → models/alphabet_model.pkl')
print(f'   (Switch to MLP anytime by copying alphabet_model_mlp.pkl → alphabet_model.pkl)')
print(f'\n   Classes  : {len(le.classes_)}')
print(f'   Features : {X.shape[1]}-dim')
print('\n▶ Next: run Step 3 to start the FastAPI server!')

✅ Saved SVM  → models/alphabet_model_svm.pkl  (97.6%)
✅ Saved MLP  → models/alphabet_model_mlp.pkl  (98.7%)

🎯 Default model set to SVM → models/alphabet_model.pkl
   (Switch to MLP anytime by copying alphabet_model_mlp.pkl → alphabet_model.pkl)

   Classes  : 26
   Features : 63-dim

▶ Next: run Step 3 to start the FastAPI server!


In [18]:
import pickle
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
import uvicorn
import threading

app = FastAPI(title="ISL.learn API")

# ── CORS ──────────────────────────────────────────────────────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://localhost:3000", "http://127.0.0.1:5173"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load BOTH models ──────────────────────────────────────────────────────
def load_model(path):
    try:
        with open(path, 'rb') as f:
            m = pickle.load(f)
        print(f'  ✅ Loaded {m["model_type"]} from {path} — {m["accuracy"]*100:.1f}%')
        return m
    except FileNotFoundError:
        print(f'  ⚠  Not found: {path}')
        return None

print('Loading models...')
_svm = load_model('models/alphabet_model_svm.pkl')
_mlp = load_model('models/alphabet_model_mlp.pkl')

# ✏️ SWITCH MODEL HERE — change to 'mlp' to use MLP instead
ACTIVE_MODEL = 'svm'

def get_active():
    return _svm if ACTIVE_MODEL == 'svm' else _mlp

# ── Normalise (same as training) ──────────────────────────────────────────
def normalize(kps: np.ndarray) -> np.ndarray:
    pts   = kps.reshape(21, 3)
    pts   = pts - pts[0]
    scale = np.linalg.norm(pts[9])
    if scale > 0:
        pts /= scale
    return pts.flatten().astype(np.float32)

# ── Input schema ──────────────────────────────────────────────────────────
class KeypointInput(BaseModel):
    keypoints: List[float]   # 63 floats
    mode: str = 'alphabet'

# ── Predict endpoint ──────────────────────────────────────────────────────
@app.post('/predict/alphabet')
async def predict_alphabet(body: KeypointInput):
    alpha = get_active()
    if alpha is None:
        return {'error': 'Model not loaded', 'code': 503}
    if len(body.keypoints) != 63:
        return {'error': f'Expected 63 keypoints, got {len(body.keypoints)}'}

    kps      = normalize(np.array(body.keypoints, dtype=np.float32))
    model    = alpha['model']
    le       = alpha['label_encoder']
    proba    = model.predict_proba([kps])[0]
    top5_idx = np.argsort(proba)[::-1][:5]

    return {
        'label'      : le.inverse_transform([top5_idx[0]])[0],
        'confidence' : float(proba[top5_idx[0]]),
        'model_used' : alpha['model_type'],
        'top5'       : [
            {'label': le.inverse_transform([i])[0], 'conf': float(proba[i])}
            for i in top5_idx
        ]
    }

# ── Switch model endpoint (no restart needed!) ────────────────────────────
@app.post('/switch/{model_name}')
async def switch_model(model_name: str):
    global ACTIVE_MODEL
    if model_name not in ('svm', 'mlp'):
        return {'error': 'Use /switch/svm or /switch/mlp'}
    ACTIVE_MODEL = model_name
    active = get_active()
    return {
        'switched_to' : model_name.upper(),
        'accuracy'    : f"{active['accuracy']*100:.1f}%" if active else 'N/A'
    }

# ── Health check ──────────────────────────────────────────────────────────
@app.get('/health')
def health():
    return {
        'status'       : 'ok',
        'active_model' : ACTIVE_MODEL.upper(),
        'svm_accuracy' : f"{_svm['accuracy']*100:.1f}%" if _svm else 'not loaded',
        'mlp_accuracy' : f"{_mlp['accuracy']*100:.1f}%" if _mlp else 'not loaded',
    }

# ── Start server ──────────────────────────────────────────────────────────
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

active = get_active()
print(f'\n🚀 Server running at http://localhost:8000')
print(f'   Active model : {ACTIVE_MODEL.upper()} ({active["accuracy"]*100:.1f}% accuracy)')
print(f'   Switch model : POST http://localhost:8000/switch/svm  or  /switch/mlp')
print(f'   Health check : http://localhost:8000/health')
print(f'\n⏳ Keep this cell running while using the website...')

Loading models...
  ✅ Loaded SVM from models/alphabet_model_svm.pkl — 97.6%
  ✅ Loaded MLP from models/alphabet_model_mlp.pkl — 98.7%

🚀 Server running at http://localhost:8000
   Active model : SVM (97.6% accuracy)
   Switch model : POST http://localhost:8000/switch/svm  or  /switch/mlp
   Health check : http://localhost:8000/health

⏳ Keep this cell running while using the website...


INFO:     Started server process [3721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
